# 作业 01：用 PyTorch 学习参数 w

本次任务：让计算机调整一个数 **w**，使预测值越来越接近目标值。

$$\hat y=xw^2,\qquad L=|y-xw^2|.$$

完成后，你应该能够：区分数据和待学习参数；使用 `backward()` 计算梯度；使用优化器更新参数；解释初始化和学习率为什么会影响训练。

**使用方法：**先在终端激活上一份作业的 `torch-class` 环境，再运行 `python -m notebook`。打开本文件，选择 **Python (torch-class)** 内核，从上到下按 **Shift + Enter**。`conda activate` 是终端命令，不要写进下面的 Python 单元格。

本作业只需要 PyTorch，使用 CPU，不需要显卡或额外数据文件。提交前请重启内核并运行全部单元格，保存代码与输出。

In [1]:
# 检查 Notebook 实际使用的 Python 和 PyTorch
import sys
import torch

print("Python 路径：", sys.executable)
print("PyTorch 版本：", torch.__version__)

C:\ProgramData\miniconda3\envs\torch-class\Lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Python 路径： C:\ProgramData\miniconda3\envs\torch-class\python.exe
PyTorch 版本： 2.7.1+cpu


## 1. 什么已知，什么需要学习？

只使用一个样本：`x=1`，`y=4`。它们在训练中保持不变，只有 `w` 会改变。

预测公式为 $\hat y=xw^2$，不是 $xw$。这里 `** 2` 表示平方，`torch.abs(...)` 表示绝对值。

先手算：当 `w=1` 时，预测值为 1，损失为 3。要让损失为 0，需要 $w^2=4$，因此 **w=2 和 w=-2 都是正确解**。训练只能通过这个样本确定平方的值，不能唯一确定正负号。

In [2]:
# 输入和目标是固定数据，不需要求梯度
x = torch.tensor(1.0)
y = torch.tensor(4.0)

# 使用浮点数；requires_grad=True 让 PyTorch 跟踪与 w 有关的计算
w = torch.tensor(1.0, requires_grad=True)
y_pred = x * w ** 2
loss = torch.abs(y - y_pred)

print("当前 w：", w.item())
print("预测值：", y_pred.item())
print("损失：", loss.item())

当前 w： 1.0
预测值： 1.0
损失： 3.0


## 2. 先看一次梯度和更新

梯度告诉我们：在当前位置，`w` 稍微增大时，损失会怎样变化。更新规则是：

$$w_{\text{新}}=w_{\text{旧}}-\text{学习率}\times\text{梯度}.$$

本例从 `w=1` 开始，梯度为 `-2`。学习率为 `0.1` 时，下一步应得到 `w=1.2`。

`backward()` 计算梯度；`step()` 根据梯度更新参数；`zero_grad()` 清除旧梯度，不会把 `w` 重置为 0。`requires_grad=True` 本身不会开始训练。

In [3]:
# 本单元格每次都会重新创建 w，可以独立重复运行
w = torch.tensor(1.0, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)

optimizer.zero_grad()
loss = torch.abs(y - x * w ** 2)
loss.backward()
print("更新前的 w：", w.item())
print("更新前的损失：", loss.item())
print("梯度：", w.grad.item())

optimizer.step()
# 参数更新后重新计算损失，让打印的 w 和损失属于同一时刻
with torch.no_grad():
    new_loss = torch.abs(y - x * w ** 2)
print("更新后的 w：", w.item())
print("更新后的损失：", new_loss.item())

更新前的 w： 1.0
更新前的损失： 3.0
梯度： -2.0
更新后的 w： 1.2000000476837158
更新后的损失： 2.559999942779541


## 3. 重复更新，让 w 接近一个解

绝对值损失在误差为 0 处有尖角。固定学习率可能让 `w` 在解的两侧来回跳动，损失也不保证每一步都下降。下面使用逐渐减小的学习率 `0.1 / 轮数`，让后面的调整幅度变小。

这是一种便于观察的教学选择，不保证任意数据或初始化都能训练成功。我们用 `torch.no_grad()` 记录更新后的结果，不为这些记录额外构建求导过程。

In [4]:
def train_w(initial_w=1.0, epochs=1000, base_lr=0.1, decay=True):
    # 每次调用都从指定初值开始，避免上次实验的参数影响本次结果
    w = torch.tensor(float(initial_w), requires_grad=True)
    optimizer = torch.optim.SGD([w], lr=base_lr)
    history = []

    for epoch in range(1, epochs + 1):
        lr = base_lr / epoch if decay else base_lr
        optimizer.param_groups[0]["lr"] = lr
        optimizer.zero_grad()
        y_pred = x * w ** 2
        loss = torch.abs(y - y_pred)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            prediction = (x * w ** 2).item()
            current_loss = torch.abs(y - x * w ** 2).item()
        history.append((epoch, lr, w.item(), prediction, current_loss))
    return w.detach().item(), history

learned_w, history = train_w()
print("轮数    学习率          w           预测值        损失")
for index in [0, 1, 4, 9, 99, 999]:
    epoch, lr, value, prediction, current_loss = history[index]
    print(f"{epoch:4d}  {lr:.6f}  {value:11.6f}  {prediction:11.6f}  {current_loss:.6f}")
print("最终学到的 w：", learned_w)

轮数    学习率          w           预测值        损失
   1  0.100000     1.200000     1.440000  2.560000
   2  0.050000     1.320000     1.742400  2.257600
   5  0.020000     1.537536     2.364017  1.635983
  10  0.010000     1.746504     3.050276  0.949724
 100  0.001000     1.999979     3.999917  0.000083
1000  0.000100     2.000000     4.000000  0.000000
最终学到的 w： 2.0


## 4. 为什么不能从 0 开始？

在误差不为 0 时，损失对 `w` 的梯度为：

$$\frac{\partial L}{\partial w}=2xw\,\mathrm{sign}(xw^2-y).$$

不需要背公式，观察其中乘了 `w`：**当 w=0 时，梯度也是 0**。即使损失很大，梯度下降仍会停在这里。

在误差恰好为 0 的尖角处，普通导数不存在；PyTorch 对绝对值在 0 处采用的梯度为 0。

先预测下面三个实验的结果，再运行：正数初值、负数初值、零初值分别会怎样？

In [5]:
for initial_w in [1.0, -1.0, 0.0]:
    final_w, records = train_w(initial_w=initial_w)
    print(f"初始 w={initial_w:5.1f} → 最终 w={final_w: .6f}，最终损失={records[-1][-1]:.6f}")

初始 w=  1.0 → 最终 w= 2.000000，最终损失=0.000000
初始 w= -1.0 → 最终 w=-2.000000，最终损失=0.000000
初始 w=  0.0 → 最终 w= 0.000000，最终损失=4.000000


## 5. 对比：固定学习率会怎样？

下面仍使用完全相同的损失，只更换学习率策略。观察最后 8 轮：`w` 是否跨过 2？损失是否每轮都下降？不要求每一轮的结果完全一致，浮点数和软件版本可能造成小差异。

In [6]:
_, fixed_history = train_w(epochs=40, base_lr=0.1, decay=False)
print("轮数       w           损失")
for epoch, lr, value, prediction, current_loss in fixed_history[-8:]:
    print(f"{epoch:4d}  {value:11.6f}  {current_loss:.6f}")

轮数       w           损失
  33     2.107632  0.442114
  34     1.686106  1.157047
  35     2.023327  0.093853
  36     1.618662  1.379935
  37     1.942394  0.227105
  38     2.330873  1.432968
  39     1.864698  0.522901
  40     2.237638  1.007023


## 6. 你的作业

1. 用自己的话说明：`x`、`y`、`w`、`y_pred`、`loss` 分别是什么？哪些在训练中改变？
2. 将下面实验的初始值改为 `0.5` 和 `-0.5`，分别运行并记录最终参数和损失。两次预测是否都接近 4？
3. 将学习率改为 `0.01`，比较相同轮数下的结果。学习率更小是否一定更好？请结合输出说明。
4. 解释为什么 `w=0` 的实验没有学到正确答案，以及为什么 `w=-2` 也是正确答案。
5. 选做：把上面的 `y` 改成 `9.0`，从定义数据的单元格开始重新运行。先写出两个理论解，再检查训练结果。

请在下方文字单元格写出观察和解释，不能只提交运行输出。

In [7]:
# ===== 作业实验部分 =====

# 实验 2：初始值 0.5 和 -0.5 的对比
print("=" * 50)
print("实验 2：不同初始值对比")
print("=" * 50)
for initial_w in [0.5, -0.5]:
    my_w, my_history = train_w(initial_w=initial_w, base_lr=0.1)
    print(f"初始 w = {initial_w:5.1f}")
    print(f"  最终 w     = {my_w:.6f}")
    print(f"  最终预测值 = {my_history[-1][-2]:.6f}")
    print(f"  最终损失   = {my_history[-1][-1]:.6f}")
    print()

# 实验 3：学习率 0.01 对比
print("=" * 50)
print("实验 3：学习率对比（初始 w=1.0）")
print("=" * 50)
for lr in [0.1, 0.01]:
    lr_w, lr_history = train_w(initial_w=1.0, base_lr=lr)
    print(f"学习率 base_lr = {lr:.2f}")
    print(f"  最终 w     = {lr_w:.6f}")
    print(f"  最终预测值 = {lr_history[-1][-2]:.6f}")
    print(f"  最终损失   = {lr_history[-1][-1]:.6f}")
    print()

# 选做实验：y = 9.0
print("=" * 50)
print("选做实验：y = 9.0")
print("=" * 50)
print("理论解：w² = 9 → w = 3 或 w = -3")
print()

# 临时修改 y 来测试
y_original = y
y = torch.tensor(9.0)
for initial_w in [1.0, -1.0]:
    test_w, test_history = train_w(initial_w=initial_w)
    print(f"初始 w = {initial_w:5.1f}")
    print(f"  最终 w     = {test_w:.6f}")
    print(f"  最终预测值 = {test_history[-1][-2]:.6f}")
    print(f"  最终损失   = {test_history[-1][-1]:.6f}")
    print()
# 恢复 y
y = y_original


实验 2：不同初始值对比
初始 w =   0.5
  最终 w     = 2.000117
  最终预测值 = 4.000469
  最终损失   = 0.000469

初始 w =  -0.5
  最终 w     = -2.000117
  最终预测值 = 4.000469
  最终损失   = 0.000469

实验 3：学习率对比（初始 w=1.0）
学习率 base_lr = 0.10
  最终 w     = 2.000000
  最终预测值 = 4.000000
  最终损失   = 0.000000

学习率 base_lr = 0.01
  最终 w     = 1.161119
  最终预测值 = 1.348197
  最终损失   = 2.651803

选做实验：y = 9.0
理论解：w² = 9 → w = 3 或 w = -3

初始 w =   1.0
  最终 w     = 3.000000
  最终预测值 = 9.000000
  最终损失   = 0.000000

初始 w =  -1.0
  最终 w     = -3.000000
  最终预测值 = 9.000000
  最终损失   = 0.000000



### 我的回答

**1. 各个变量的含义：**

- **x**：输入数据，固定为 1.0，是我们已知的样本特征，训练过程中不改变。
- **y**：目标值（标签），固定为 4.0，是我们希望预测值逼近的目标，训练过程中不改变。
- **w**：待学习的参数，是我们要训练调整的变量，训练过程中不断变化。
- **y_pred**：预测值，由公式 `x * w²` 计算得到，会随着 w 的变化而变化。
- **loss**：损失值，衡量预测值和目标值的差距，这里用绝对值 `|y - y_pred|`，训练过程中不断变化。

**训练中会改变的：** w、y_pred、loss。
**训练中保持不变的：** x、y。

---

**2. 正、负初值的对比（初始 w=0.5 和 w=-0.5）：**

运行结果：
- 初始 w = 0.5 → 最终 w ≈ 2.0001，最终预测 ≈ 4.0005，最终损失 ≈ 0.0005
- 初始 w = -0.5 → 最终 w ≈ -2.0001，最终预测 ≈ 4.0005，最终损失 ≈ 0.0005

两次预测都接近 4，损失都接近 0。区别在于：正数初值收敛到了正解 w=2，负数初值收敛到了负解 w=-2。因为 w 经过平方运算后，正负号不影响预测结果。

---

**3. 学习率实验的观察：**

对比 base_lr=0.1 和 base_lr=0.01（初始 w=1.0，训练 1000 轮）：
- base_lr=0.1 → 最终 w ≈ 2.0000，最终预测 = 4.0000，最终损失 ≈ 0.0000
- base_lr=0.01 → 最终 w ≈ 1.1611，最终预测 ≈ 1.3482，最终损失 ≈ 2.6518

**学习率更小不一定更好。** 这里 base_lr=0.01 的学习率太小了，而且学习率还会随轮数衰减（lr = base_lr / epoch），到后面步长变得极小，导致 w 移动得非常慢，1000 轮下来根本走不到 w=2 的位置，损失还有 2.65，完全没收敛。

学习率太小会导致收敛太慢甚至不收敛；学习率太大又可能导致震荡不稳定。合适的学习率需要在收敛速度和稳定性之间取得平衡。

---

**4. 零初始化与两个解的解释：**

**为什么 w=0 没有学到正确答案？**
梯度公式里有 w 这个因子（∂L/∂w = 2xw·sign(...)），当 w=0 时，梯度也是 0。即使损失很大（这里损失是 4），梯度下降的更新量 = 学习率 × 梯度 = 0，w 永远停在 0 不动，所以学不到正确答案。这就是"零初始化"在这种带乘法参数的模型里会卡住的原因。

**为什么 w=-2 也是正确答案？**
因为预测公式是 ŷ = x·w²，w 经过平方运算。(-2)² = 4，和 2² = 4 的结果完全一样，所以 w=2 和 w=-2 都能让预测值等于 4，损失都是 0。从这个样本数据来看，我们只能确定 w²=4，但无法判断 w 应该是正还是负——正负号是对称的，数据本身没有提供区分正负的信息。

---

**5. 选做实验（y=9.0）：**

理论解：由 w² = 9 得 w = 3 或 w = -3。

运行结果：
- 初始 w = 1.0 → 最终 w ≈ 3.0000，最终预测 = 9.0000，最终损失 ≈ 0.0000
- 初始 w = -1.0 → 最终 w ≈ -3.0000，最终预测 = 9.0000，最终损失 ≈ 0.0000

实验结果和理论预测一致：正数初值收敛到正解 w=3，负数初值收敛到负解 w=-3，两个解都能让预测值准确等于 9。

---

### 提交前检查

- [x] 已选择自己的课程 Python 内核。
- [x] 已填写文字回答，并保存实验结果。
- [x] 已重启内核并从上到下运行全部单元格，没有报错。
- [x] 已将修改后的 `.ipynb` 保存并提交到自己的 GitHub 仓库。
